In [15]:
import pandas as pd
import os
import re
from pymongo import MongoClient
from collections import OrderedDict

# --- Step 1: Load Target.xlsx ---
target_file = r"D:\DevGo\Work\MDM3_Team\Target_files\Target_BATCH_1_500_AHFL.xlsx"
target_ids = pd.read_excel(target_file).iloc[:, 0].dropna().astype(str)
target_filenames = ["AHFL_" + id_ + ".TIF" for id_ in target_ids]

# --- Step 2: Connect to MongoDB ---
client = MongoClient("mongodb://172.19.19.87:27017/")
db = client["AHFL_Full"]
collection = db["charts_2"]

# --- Step 3: Query only required documents ---
docs = collection.find(
    {"file_name": {"$in": target_filenames}},
    {"file_name": 1, "visits": 1}
)

# --- Step 4: Normalize visit type ---
def normalize_visit_type(raw_type: str) -> str:
    raw_type = raw_type.lower()
    if "h&p" in raw_type or "h and p" in raw_type or "h&lp" in raw_type:
        return "initial"
    elif "progress" in raw_type:
        return "subsequent"
    elif "discharge" in raw_type:
        return "discharge"
    elif "consult" in raw_type:
        return "consult"
    elif "procedure" in raw_type:
        return "procedure"
    else:
        return raw_type

# --- Step 5: Extract rows ---
data_rows = []
all_section_names = set()
sr_no = 1

for doc in docs:
    file_name = doc.get("file_name", "").replace(".tiff", "")
    # file_name = re.sub(r'\.tif+f?$', '', doc.get("file_name", ""), flags=re.IGNORECASE)

    visits = doc.get("visits", {})

    for visit_key, visit_data in visits.items():
        if visit_key.startswith("Date_of_Service_"):
            visit_info = visit_key.replace("Date_of_Service_", "")
            parts = visit_info.split()

            if len(parts) >= 2:
                dos = parts[0]
                visit_type_raw = " ".join(parts[2:]) if len(parts) > 2 else ""
                visit_type = normalize_visit_type(visit_type_raw.lower())

                section_dict = {
                    "Sr No": sr_no,
                    "Filename": file_name,
                    "DOS": dos,
                    "Visit": visit_type
                }

                if not isinstance(visit_data, list):
                    continue

                for entry in visit_data:
                    section = entry.get("section", "").strip()
                    content = entry.get("content", "")
                    if section:
                        section_dict[section] = content
                        all_section_names.add(section)

                data_rows.append(section_dict)
                sr_no += 1

# --- Step 6: Normalize rows ---
all_section_names = sorted(all_section_names)
base_columns = ["Sr No", "Filename", "DOS", "Visit"]
final_columns = base_columns + list(all_section_names)

for row in data_rows:
    for col in all_section_names:
        if col not in row:
            row[col] = ""

df = pd.DataFrame(data_rows, columns=final_columns)

# f_name = r"C:\Users\pbhopale\Desktop\Warpper_V4\All_Headers_Data\BATCH_1_AHFL_All_Header_Data_From_Mango_DB.xlsx"
f_name = r"D:\DevGo\Work\MDM3_Team\All_Headers_Data\BATCH_1_AHFL_All_Header_Data_From_Mango_DB.xlsx"
df.to_excel(f_name, index=False)

In [16]:
df


,Sr No,Filename,DOS,Visit,Assessment,Assessment & Plan,Assessment/Plan,Chief Complaint,Code Status,Current Facility-Administered Medications,...,PAST SOCIAL HX,PAST SURGICAL HISTORY,PAST SURGICAL HX,PHYSICAL EXAMINATION,PRN MEDICATIONS,Physical Exam,Result Date,Review of Systems,Significant Findings/Tests/Studies,Subjective
0,1,AHFL_HMV001970181.TIF,10/2/2024,initial,,n\nAcute anemia\nPatient reports history of ac...,,"Dizziness, weakness, visual disturbance x1 day",s: Full Code\nConsultants\nNot on file\nAdditi...,,...,reports that she has never smoked. She has nev...,,has no past surgical history on file.,,,"General: alert, awake, no acute distress\n.\nS...",: 10/2/2024\nNo acute cardiopulmonary disease....,ROS Positive as per HPI. All other systems rev...,,
1,2,AHFL_HMV001970181.TIF,10/4/2024,discharge,,,,,,,...,,,,,,patient was seen and examined\nGENERAL: Alert ...,: 10/3/2024\nMarkedly enlarged uterus with mul...,,"LAB RESULTS\n@RESUFAST (WBC, HGB, HCT,MCVPLT)@...",
2,3,AHFL_HMV001970181.TIF,10/3/2024,subsequent,,lan\nAcute anemia\nPatient reports history of ...,,,ode\nNext of kin: Husband\nPrognosis: Stable\n...,,...,,,,patient was seen and examined\nGENERAL: Alert ...,s: acetaminophen **OR** acetaminophen **OR** a...,,: 10/3/2024\nMarkedly enlarged uterus with mul...,,,%\n10/3/2024 patient was seen\nat bedside this...
3,4,AHFL_HMV001970575.TIF,10/2/2024,initial,,n\nAthscl heart disease of native cor art W ot...,,Status post cardiac catheterization,s: Full Code\nConsultants\nNot on file\nDispos...,,...,reports that she quit smoking about 32 years a...,,has a past surgical history that includes US G...,,": acetaminophen, morphine, nitroglycerin, onda...","m\nGeneral: alert, awake, no acute distress\n-...",,s\nROS Positive as per HPI. All other systems ...,,
4,5,AHFL_HMV001970575.TIF,10/3/2024,discharge,,,,,,,...,,,,,,"General: alert, awake, no acute distress.\nSki...",,,"LAB RESULTS\n@RESUFAST (WBC, HGB, HCT,MCVPLT)@...",
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2434,2435,AHFL_HMV002007126.TIF,10/20/2024,discharge,,,,,,,...,,,,,,Patient was examined on day of discharge,: 10/20/2024\nNo acute intracranial abnormalit...,,"LAB RESULTS\n@RESUFAST (WBC, HGB, HCT,MCV/PLT)...",
2435,2436,AHFL_HMV002007077.TIF,10/19/2024,initial,,an\nChest pain\nCXR completed and shows low lu...,,Chest pain at rest x1 hour,us: Full Code\nConsultants\nProvider Specialty...,,...,reports that he has never smoked. He has never...,,has a past surgical history that includes LACE...,,": heparin, Insert peripheral IV **AND** Saline...","m\nGeneral: alert, awake, no acute distress\n....",e: 10/19/2024\nExpiratory chest with low lung ...,s\nROS Positive as per HPI. All other systems ...,,
2436,2437,AHFL_HMV002007077.TIF,10/20/2024,discharge,,,,,,,...,,,,,,"General: alert, awake, no acute distress.\nSki...",,,"LAB RESULTS\n@RESUFAST (WBC, HGB, HCT,MCVPLT)@...",
2437,2438,AHFL_HMV002007415.TIF,10/19/2024,initial,,Anaphylactic reaction\nReportedly allergic rea...,,Anaphylaxis to insect bite,s: Full code\nConsultants\nNot on file\nDispos...,,...,reports that she has never smoked. She has nev...,,has a past surgical history that includes BI M...,,,"General: alert, awake, mild distress\n.\nSkin:...",,ROS Positive as per HPI. All other systems rev...,,
